In [ ]:
import random
from collections import Counter, defaultdict, deque
from functools import cmp_to_key
from itertools import zip_longest

# Arrays and Strings

---
# 1) Introduction

Linear-scan problems over arrays and strings. Most of them have an $O(n^2)$ brute force that
compares every pair or every window, and a linear solution built on one of three ideas:

| Idea | It removes | Typical cue in the statement |
|---|---|---|
| Hash map / set | the inner search loop ("have I seen X?") | pair with sum, duplicates, grouping, counting |
| Two pointers | one of two nested indices, using sortedness or symmetry | sorted input, palindrome, pair/triple sum, in-place |
| Sliding window | recomputing every substring/subarray from scratch | longest/shortest contiguous block with a property |
| Prefix sum | recomputing range sums | "subarray sum equals / divisible by k", range queries |

**How to use each question block.** Read the *Problem* line, cover the rest, and try for five
minutes. If stuck, read only the *Signal* (which pattern the wording points to), try again,
and then read the *Idea*. Run the cell; the `assert` lines are LeetCode's examples plus edge
cases. Tiers: ★★★ Blind 75 (do first), ★★ NeetCode 150 / Grind 169 / Top Interview 150,
★ other frequently asked.

---
# 2) Arrays and Hashing

A hash map turns "search the rest of the array" ($O(n)$) into a lookup ($O(1)$ average). The
question to ask is *what key do I store so that the answer becomes a lookup?* — the value
itself (Two Sum), a canonical form (sorted letters for anagrams), or a running state
(prefix sum, see §5).

## 2.1 Hash map and set lookups

### 2.1.1 [1. Two Sum](https://leetcode.com/problems/two-sum/) — ★★★ · Easy

**Problem.** Return indices of the two numbers that add up to `target` (exactly one answer).  
**Signal.** Pair with a given sum, unsorted input, indices required → hash map of value → index.  
**Idea.** Scan left to right. For each `x`, the partner is `target - x`; if it was seen already, return both indices, otherwise store `x`. One pass, and an element never pairs with itself because it is stored after the lookup.  
**Cost.** O(n) time · O(n) space

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    """Indices i < j with nums[i] + nums[j] == target."""
    seen: dict[int, int] = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return []


assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
print(two_sum([2, 7, 11, 15], 9))

### 2.1.2 [217. Contains Duplicate](https://leetcode.com/problems/contains-duplicate/) — ★★★ · Easy

**Problem.** Return `True` if any value appears at least twice.  
**Signal.** "Seen before?" → set.  
**Idea.** Insert each value into a set; a hit means a duplicate. (One-liner: `len(set(nums)) < len(nums)`, but the loop exits early.)  
**Cost.** O(n) time · O(n) space

In [ ]:
def contains_duplicate(nums: list[int]) -> bool:
    """True if any value repeats."""
    seen: set[int] = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False


assert contains_duplicate([1, 2, 3, 1])
assert not contains_duplicate([1, 2, 3, 4])
print(contains_duplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]))

### 2.1.3 [242. Valid Anagram](https://leetcode.com/problems/valid-anagram/) — ★★★ · Easy

**Problem.** Is `t` a rearrangement of `s`?  
**Signal.** Same multiset of characters → compare counts.  
**Idea.** Two strings are anagrams iff their character counts are identical. `Counter` equality does it; a fixed array of 26 counts is the same idea without hashing.  
**Cost.** O(n) time · O(1) space (26 letters)

In [ ]:
def is_anagram(s: str, t: str) -> bool:
    """True if t is a permutation of s."""
    return len(s) == len(t) and Counter(s) == Counter(t)


assert is_anagram("anagram", "nagaram")
assert not is_anagram("rat", "car")
print(is_anagram("listen", "silent"))

### 2.1.4 [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/) — ★★★ · Medium

**Problem.** Group the strings that are anagrams of each other.  
**Signal.** Grouping by an equivalence → hash map from a canonical key to a list.  
**Idea.** All anagrams share the same sorted string (or the same 26-count tuple). Use that as the key and append each word to its bucket.  
**Cost.** O(n·k log k) with sorted key, O(n·k) with count key · O(n·k) space  
**Trap.** Lists are not hashable — convert the count array to a `tuple`.

In [ ]:
def group_anagrams(strs: list[str]) -> list[list[str]]:
    """Buckets of mutually anagrammatic words."""
    groups: dict[tuple[int, ...], list[str]] = defaultdict(list)
    for word in strs:
        count = [0] * 26
        for ch in word:
            count[ord(ch) - ord("a")] += 1
        groups[tuple(count)].append(word)
    return list(groups.values())


out = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(map(sorted, out)) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
print(out)

### 2.1.5 [347. Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/) — ★★★ · Medium

**Problem.** Return the `k` most frequent values.  
**Signal.** "Top k" by frequency → count, then bucket sort (or heap).  
**Idea.** Count with a hash map. A frequency is at most `n`, so put each value in bucket `freq` and read buckets from high to low until `k` values are collected — linear time, beating the $O(n \log k)$ heap.  
**Cost.** O(n) time · O(n) space

In [ ]:
def top_k_frequent(nums: list[int], k: int) -> list[int]:
    """k values with the highest counts."""
    buckets: list[list[int]] = [[] for _ in range(len(nums) + 1)]
    for value, freq in Counter(nums).items():
        buckets[freq].append(value)
    out: list[int] = []
    for freq in range(len(buckets) - 1, 0, -1):
        out.extend(buckets[freq])
        if len(out) >= k:
            return out[:k]
    return out


assert sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
print(top_k_frequent([4, 4, 4, 5, 5, 6], 2))

### 2.1.6 [238. Product of Array Except Self](https://leetcode.com/problems/product-of-array-except-self/) — ★★★ · Medium

**Problem.** `out[i]` = product of every element except `nums[i]`, without division, in O(n).  
**Signal.** "Everything except i" → prefix product × suffix product.  
**Idea.** `out[i] = (product of left part) × (product of right part)`. First pass writes the left products into `out`; second pass walks from the right carrying a running suffix product and multiplies it in.  
**Cost.** O(n) time · O(1) extra space (output excluded)

In [ ]:
def product_except_self(nums: list[int]) -> list[int]:
    """Product of all other elements for each index."""
    n = len(nums)
    out = [1] * n
    for i in range(1, n):
        out[i] = out[i - 1] * nums[i - 1]
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out


assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]
assert product_except_self([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0]
print(product_except_self([1, 2, 3, 4]))

### 2.1.7 [128. Longest Consecutive Sequence](https://leetcode.com/problems/longest-consecutive-sequence/) — ★★★ · Medium

**Problem.** Length of the longest run of consecutive integers (any order), in O(n).  
**Signal.** O(n) forbids sorting → set + only start counting at sequence starts.  
**Idea.** Put all numbers in a set. A number `x` starts a run only if `x - 1` is absent; from each start, walk `x+1, x+2, …`. Every number is visited at most twice, so the total is linear.  
**Cost.** O(n) time · O(n) space

In [ ]:
def longest_consecutive(nums: list[int]) -> int:
    """Length of the longest consecutive-integer run."""
    values = set(nums)
    best = 0
    for x in values:
        if x - 1 not in values:
            length = 1
            while x + length in values:
                length += 1
            best = max(best, length)
    return best


assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive([]) == 0
print(longest_consecutive([100, 4, 200, 1, 3, 2]))

### 2.1.8 [271. Encode and Decode Strings](https://leetcode.com/problems/encode-and-decode-strings/) — ★★★ · Medium · Premium

**Problem.** Serialise a list of strings into one string and back; strings may contain any character.  
**Signal.** Arbitrary content → length-prefix framing (a delimiter alone can be ambiguous).  
**Idea.** Write each string as `len#text`. To decode, read digits up to `#`, then take exactly that many characters. The `#` inside a string is harmless because we never search for it past the length header.  
**Cost.** O(total length) time and space

In [ ]:
def encode(strs: list[str]) -> str:
    """Length-prefixed concatenation."""
    return "".join(f"{len(s)}#{s}" for s in strs)


def decode(data: str) -> list[str]:
    """Inverse of encode."""
    out: list[str] = []
    i = 0
    while i < len(data):
        j = data.index("#", i)
        length = int(data[i:j])
        out.append(data[j + 1 : j + 1 + length])
        i = j + 1 + length
    return out


for case in [["lint", "code", "love", "you"], ["", "a#b", "12#"], []]:
    assert decode(encode(case)) == case
print(encode(["a#b", ""]))

### 2.1.9 [36. Valid Sudoku](https://leetcode.com/problems/valid-sudoku/) — ★★ · Medium

**Problem.** Check a partially filled 9×9 board: no repeated digit in any row, column or 3×3 box.  
**Signal.** Several "no duplicates within group" constraints → one set per group.  
**Idea.** For every filled cell, the box index is `(r // 3, c // 3)`. Keep sets for rows, columns and boxes; a digit already present in any of its three sets makes the board invalid.  
**Cost.** O(81) time and space

In [ ]:
def is_valid_sudoku(board: list[list[str]]) -> bool:
    """True if no row/column/box repeats a digit."""
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                continue
            b = (r // 3) * 3 + c // 3
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                return False
            rows[r].add(d)
            cols[c].add(d)
            boxes[b].add(d)
    return True


board = [
    list("53..7...."),
    list("6..195..."),
    list(".98....6."),
    list("8...6...3"),
    list("4..8.3..1"),
    list("7...2...6"),
    list(".6....28."),
    list("...419..5"),
    list("....8..79"),
]
assert is_valid_sudoku(board)
board[0][0] = "8"
assert not is_valid_sudoku(board)
print("ok")

### 2.1.10 [383. Ransom Note](https://leetcode.com/problems/ransom-note/) — ★★ · Easy

**Problem.** Can `ransom_note` be built from the letters of `magazine` (each letter used once)?  
**Signal.** Multiset containment → counts.  
**Idea.** Count the magazine letters, then consume one for each note letter; failing to find one means no. `Counter` subtraction expresses it in one line.  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def can_construct(ransom_note: str, magazine: str) -> bool:
    """True if note letters are a sub-multiset of magazine letters."""
    return not Counter(ransom_note) - Counter(magazine)


assert not can_construct("a", "b")
assert not can_construct("aa", "ab")
assert can_construct("aa", "aab")
print(can_construct("aa", "aab"))

### 2.1.11 [205. Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) — ★★ · Easy

**Problem.** Can characters of `s` be replaced to get `t` with a one-to-one mapping?  
**Signal.** Bijection between two alphabets → two maps (or compare first-occurrence patterns).  
**Idea.** Map each `s` char to a `t` char and each `t` char back to an `s` char; any conflict breaks the bijection. Equivalent trick: both strings must have the same "index of first occurrence" sequence.  
**Cost.** O(n) time · O(1) space

In [ ]:
def is_isomorphic(s: str, t: str) -> bool:
    """True if a bijective char mapping turns s into t."""
    return [s.index(c) for c in s] == [t.index(c) for c in t]


assert is_isomorphic("egg", "add")
assert not is_isomorphic("foo", "bar")
assert not is_isomorphic("badc", "baba")
print(is_isomorphic("paper", "title"))

### 2.1.12 [290. Word Pattern](https://leetcode.com/problems/word-pattern/) — ★★ · Easy

**Problem.** Does the word list follow the letter `pattern` bijectively (`"abba"` ↔ `"dog cat cat dog"`)?  
**Signal.** Same bijection test as Isomorphic Strings, with words instead of characters.  
**Idea.** Lengths must match, then the letter→word and word→letter maps must both be consistent. Comparing the number of distinct letters, words and pairs is a compact way to say that.  
**Cost.** O(n) time · O(n) space

In [ ]:
def word_pattern(pattern: str, s: str) -> bool:
    """True if words map one-to-one onto pattern letters."""
    words = s.split()
    if len(words) != len(pattern):
        return False
    return len(set(pattern)) == len(set(words)) == len(set(zip(pattern, words)))


assert word_pattern("abba", "dog cat cat dog")
assert not word_pattern("abba", "dog cat cat fish")
assert not word_pattern("abba", "dog dog dog dog")
print(word_pattern("abba", "dog cat cat dog"))

### 2.1.13 [219. Contains Duplicate II](https://leetcode.com/problems/contains-duplicate-ii/) — ★★ · Easy

**Problem.** Are there `i != j` with `nums[i] == nums[j]` and `|i - j| <= k`?  
**Signal.** Duplicate within distance k → map value → last index (or a size-k window set).  
**Idea.** Remember the latest index of each value. When the value reappears, check the gap to the latest index — the latest one is the closest candidate.  
**Cost.** O(n) time · O(n) space

In [ ]:
def contains_nearby_duplicate(nums: list[int], k: int) -> bool:
    """True if equal values occur within k positions."""
    last: dict[int, int] = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False


assert contains_nearby_duplicate([1, 2, 3, 1], 3)
assert contains_nearby_duplicate([1, 0, 1, 1], 1)
assert not contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2)
print("ok")

### 2.1.14 [169. Majority Element](https://leetcode.com/problems/majority-element/) — ★★ · Easy

**Problem.** Return the element that appears more than ⌊n/2⌋ times (guaranteed to exist).  
**Signal.** Strict majority + O(1) space → Boyer–Moore voting.  
**Idea.** Pair off different elements and cancel them. Keep a `candidate` and `count`: same value increments, different value decrements, and at zero switch candidate. The majority cannot be fully cancelled, so it is the survivor.  
**Cost.** O(n) time · O(1) space

In [ ]:
def majority_element(nums: list[int]) -> int:
    """Value appearing more than n // 2 times."""
    candidate, count = 0, 0
    for x in nums:
        if count == 0:
            candidate = x
        count += 1 if x == candidate else -1
    return candidate


assert majority_element([3, 2, 3]) == 3
assert majority_element([2, 2, 1, 1, 1, 2, 2]) == 2
print(majority_element([2, 2, 1, 1, 1, 2, 2]))

### 2.1.15 [229. Majority Element II](https://leetcode.com/problems/majority-element-ii/) — ★ · Medium

**Problem.** Return all values appearing more than ⌊n/3⌋ times.  
**Signal.** Generalised Boyer–Moore: at most k−1 elements can exceed n/k.  
**Idea.** Keep two candidates and two counts. A value that matches neither and finds no empty slot decrements both (cancelling a triple of distinct values). Survivors are only *candidates*, so verify with a second counting pass.  
**Cost.** O(n) time · O(1) space

In [ ]:
def majority_element_ii(nums: list[int]) -> list[int]:
    """All values with count > n // 3."""
    c1, c2, n1, n2 = None, None, 0, 0
    for x in nums:
        if x == c1:
            n1 += 1
        elif x == c2:
            n2 += 1
        elif n1 == 0:
            c1, n1 = x, 1
        elif n2 == 0:
            c2, n2 = x, 1
        else:
            n1 -= 1
            n2 -= 1
    return [c for c in {c1, c2} if c is not None and nums.count(c) > len(nums) // 3]


assert majority_element_ii([3, 2, 3]) == [3]
assert sorted(majority_element_ii([1, 2])) == [1, 2]
assert majority_element_ii([1]) == [1]
print(majority_element_ii([1, 1, 1, 3, 3, 2, 2, 2]))

### 2.1.16 [409. Longest Palindrome](https://leetcode.com/problems/longest-palindrome/) — ★★ · Easy

**Problem.** Length of the longest palindrome that can be *built* from the letters of `s`.  
**Signal.** Palindrome from a multiset → pairs plus at most one odd centre.  
**Idea.** Every pair of equal letters can go on both sides. Use `count // 2 * 2` of each letter, and add 1 if any letter has an odd count (it sits in the middle).  
**Cost.** O(n) time · O(1) space

In [ ]:
def longest_palindrome(s: str) -> int:
    """Max palindrome length buildable from s's letters."""
    counts = Counter(s).values()
    paired = sum(c // 2 * 2 for c in counts)
    return paired + (1 if any(c % 2 for c in counts) else 0)


assert longest_palindrome("abccccdd") == 7
assert longest_palindrome("a") == 1
print(longest_palindrome("abccccdd"))

### 2.1.17 [13. Roman to Integer](https://leetcode.com/problems/roman-to-integer/) — ★★ · Easy

**Problem.** Convert a Roman numeral to an integer.  
**Signal.** Subtractive notation → compare each symbol with the next one.  
**Idea.** Add each symbol's value, except subtract it when a larger symbol follows it (`IV`, `XC`, `CM`).  
**Cost.** O(n) time · O(1) space

In [ ]:
ROMAN = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


def roman_to_int(s: str) -> int:
    """Integer value of a Roman numeral."""
    total = 0
    for i, ch in enumerate(s):
        if i + 1 < len(s) and ROMAN[ch] < ROMAN[s[i + 1]]:
            total -= ROMAN[ch]
        else:
            total += ROMAN[ch]
    return total


assert roman_to_int("III") == 3
assert roman_to_int("LVIII") == 58
assert roman_to_int("MCMXCIV") == 1994
print(roman_to_int("MCMXCIV"))

### 2.1.18 [12. Integer to Roman](https://leetcode.com/problems/integer-to-roman/) — ★★ · Medium

**Problem.** Convert an integer (1–3999) to a Roman numeral.  
**Signal.** Greedy over a fixed denomination table that already includes the subtractive pairs.  
**Idea.** List values high to low including `900, 400, 90, 40, 9, 4`; repeatedly take the largest value that fits. Including the subtractive forms in the table removes every special case.  
**Cost.** O(1) time (bounded input)

In [ ]:
ROMAN_TABLE = [
    (1000, "M"),
    (900, "CM"),
    (500, "D"),
    (400, "CD"),
    (100, "C"),
    (90, "XC"),
    (50, "L"),
    (40, "XL"),
    (10, "X"),
    (9, "IX"),
    (5, "V"),
    (4, "IV"),
    (1, "I"),
]


def int_to_roman(num: int) -> str:
    """Roman numeral for num."""
    out = []
    for value, symbol in ROMAN_TABLE:
        count, num = divmod(num, value)
        out.append(symbol * count)
    return "".join(out)


assert int_to_roman(3749) == "MMMDCCXLIX"
assert int_to_roman(58) == "LVIII"
assert int_to_roman(1994) == "MCMXCIV"
print(int_to_roman(1994))

### 2.1.19 [274. H-Index](https://leetcode.com/problems/h-index/) — ★★ · Medium

**Problem.** Largest `h` such that the researcher has at least `h` papers with ≥ `h` citations each.  
**Signal.** Threshold on counts bounded by n → counting sort (or sort descending).  
**Idea.** Citations above `n` behave like `n`, so bucket them into `0..n`. Walk `h` from `n` down, accumulating how many papers have ≥ `h` citations; the first `h` where that count reaches `h` is the answer.  
**Cost.** O(n) time · O(n) space

In [ ]:
def h_index(citations: list[int]) -> int:
    """Researcher's h-index."""
    n = len(citations)
    buckets = [0] * (n + 1)
    for c in citations:
        buckets[min(c, n)] += 1
    at_least = 0
    for h in range(n, -1, -1):
        at_least += buckets[h]
        if at_least >= h:
            return h
    return 0


assert h_index([3, 0, 6, 1, 5]) == 3
assert h_index([1, 3, 1]) == 1
assert h_index([0]) == 0
print(h_index([3, 0, 6, 1, 5]))

### 2.1.20 [41. First Missing Positive](https://leetcode.com/problems/first-missing-positive/) — ★★ · Hard

**Problem.** Smallest missing positive integer, O(n) time and O(1) extra space.  
**Signal.** Answer lies in `1..n+1` and values can index the array → cyclic sort / index marking.  
**Idea.** Place each value `v` in `1..n` at index `v-1` by swapping until every slot holds its own value or junk. Then the first index `i` with `nums[i] != i+1` gives `i+1`; if all match, answer `n+1`.  
**Cost.** O(n) time (each swap fixes one slot) · O(1) space  
**Trap.** Loop condition must check `nums[v-1] != v` to avoid infinite swaps on duplicates.

In [ ]:
def first_missing_positive(nums: list[int]) -> int:
    """Smallest positive integer not in nums."""
    n = len(nums)
    for i in range(n):
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1


assert first_missing_positive([1, 2, 0]) == 3
assert first_missing_positive([3, 4, -1, 1]) == 2
assert first_missing_positive([7, 8, 9, 11, 12]) == 1
assert first_missing_positive([1, 1]) == 2
print(first_missing_positive([3, 4, -1, 1]))

### 2.1.21 [448. Find All Numbers Disappeared in an Array](https://leetcode.com/problems/find-all-numbers-disappeared-in-an-array/) — ★ · Easy

**Problem.** Values are in `1..n`; return the ones that never appear, with O(1) extra space.  
**Signal.** Values index the array → mark presence by negating `nums[v-1]`.  
**Idea.** For each value `v` (use `abs`, it may already be negated), make `nums[v-1]` negative. Indices still holding positive numbers are the missing values.  
**Cost.** O(n) time · O(1) space

In [ ]:
def find_disappeared_numbers(nums: list[int]) -> list[int]:
    """Values in 1..n absent from nums."""
    for v in nums:
        idx = abs(v) - 1
        nums[idx] = -abs(nums[idx])
    return [i + 1 for i, v in enumerate(nums) if v > 0]


assert find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert find_disappeared_numbers([1, 1]) == [2]
print(find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]))

### 2.1.22 [454. 4Sum II](https://leetcode.com/problems/4sum-ii/) — ★ · Medium

**Problem.** Count tuples `(i, j, k, l)` with `a[i] + b[j] + c[k] + d[l] == 0` (four arrays of length n).  
**Signal.** Meet in the middle — split four loops into two halves of two.  
**Idea.** Count every pair sum of `a` and `b` in a hash map, then for every pair of `c` and `d` add the count of `-(c + d)`. Turns $O(n^4)$ into $O(n^2)$.  
**Cost.** O(n²) time · O(n²) space

In [ ]:
def four_sum_count(a: list[int], b: list[int], c: list[int], d: list[int]) -> int:
    """Number of zero-sum quadruples across four arrays."""
    ab = Counter(x + y for x in a for y in b)
    return sum(ab[-(x + y)] for x in c for y in d)


assert four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
assert four_sum_count([0], [0], [0], [0]) == 1
print(four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))

### 2.1.23 [554. Brick Wall](https://leetcode.com/problems/brick-wall/) — ★ · Medium

**Problem.** Draw a vertical line through a brick wall crossing the fewest bricks; return that count.  
**Signal.** Maximise edges aligned at the same x → count prefix-sum positions.  
**Idea.** For each row, record every internal edge position (running width, excluding the wall's right end). The best line goes through the most common edge, crossing `rows − max_edges` bricks.  
**Cost.** O(total bricks) time · O(distinct edges) space

In [ ]:
def least_bricks(wall: list[list[int]]) -> int:
    """Minimum bricks crossed by a vertical line."""
    edges: Counter[int] = Counter()
    for row in wall:
        pos = 0
        for width in row[:-1]:
            pos += width
            edges[pos] += 1
    return len(wall) - max(edges.values(), default=0)


wall = [[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]
assert least_bricks(wall) == 2
assert least_bricks([[1], [1], [1]]) == 3
print(least_bricks(wall))

## 2.2 In-place array manipulation

Many array questions forbid extra memory. The standard tool is a **write pointer**: one index
reads every element, a second index marks where the next kept element goes.

### 2.2.1 [88. Merge Sorted Array](https://leetcode.com/problems/merge-sorted-array/) — ★★ · Easy

**Problem.** Merge sorted `nums2` into sorted `nums1`, which has `n` spare slots at the end.  
**Signal.** In-place merge with free space at the back → fill from the back.  
**Idea.** Compare the largest remaining elements of both arrays and write the larger one at the last free slot. Writing from the back never overwrites an unread `nums1` element. Leftover `nums2` elements are copied; leftover `nums1` ones are already in place.  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def merge(nums1: list[int], m: int, nums2: list[int], n: int) -> None:
    """Merge nums2 into nums1 in place."""
    i, j, k = m - 1, n - 1, m + n - 1
    while j >= 0:
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[k] = nums1[i]
            i -= 1
        else:
            nums1[k] = nums2[j]
            j -= 1
        k -= 1


a = [1, 2, 3, 0, 0, 0]
merge(a, 3, [2, 5, 6], 3)
assert a == [1, 2, 2, 3, 5, 6]
b = [0]
merge(b, 0, [1], 1)
assert b == [1]
print(a)

### 2.2.2 [27. Remove Element](https://leetcode.com/problems/remove-element/) — ★★ · Easy

**Problem.** Remove all occurrences of `val` in place; return the new length.  
**Signal.** Filter in place → write pointer.  
**Idea.** Copy each element that is not `val` to `nums[write]` and advance `write`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def remove_element(nums: list[int], val: int) -> int:
    """Compact nums without val; return kept count."""
    write = 0
    for x in nums:
        if x != val:
            nums[write] = x
            write += 1
    return write


a = [0, 1, 2, 2, 3, 0, 4, 2]
k = remove_element(a, 2)
assert k == 5 and sorted(a[:k]) == [0, 0, 1, 3, 4]
print(k, a[:k])

### 2.2.3 [26. Remove Duplicates from Sorted Array](https://leetcode.com/problems/remove-duplicates-from-sorted-array/) — ★★ · Easy

**Problem.** Remove duplicates in place from a sorted array; return the number of unique values.  
**Signal.** Sorted → duplicates are adjacent → compare with last written value.  
**Idea.** Keep `nums[write - 1]` as the last kept value; write `x` only if it differs.  
**Cost.** O(n) time · O(1) space

In [ ]:
def remove_duplicates(nums: list[int]) -> int:
    """Unique-compact a sorted array in place."""
    write = 0
    for x in nums:
        if write == 0 or x != nums[write - 1]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(a)
assert a[:k] == [0, 1, 2, 3, 4]
print(k)

### 2.2.4 [80. Remove Duplicates from Sorted Array II](https://leetcode.com/problems/remove-duplicates-from-sorted-array-ii/) — ★★ · Medium

**Problem.** Same as above but keep each value at most twice.  
**Signal.** Generalises: allow `x` if it differs from the element two slots back in the output.  
**Idea.** Write `x` when `write < 2` or `x != nums[write - 2]`. If it equals the value two slots back, two copies are already kept. The same template works for "at most k".  
**Cost.** O(n) time · O(1) space

In [ ]:
def remove_duplicates_ii(nums: list[int]) -> int:
    """Keep at most two copies of each value in a sorted array."""
    write = 0
    for x in nums:
        if write < 2 or x != nums[write - 2]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 1, 2, 3, 3]
k = remove_duplicates_ii(a)
assert a[:k] == [0, 0, 1, 1, 2, 3, 3]
print(k)

### 2.2.5 [283. Move Zeroes](https://leetcode.com/problems/move-zeroes/) — ★★ · Easy

**Problem.** Move all zeros to the end, keeping the order of non-zeros, in place.  
**Signal.** Stable partition → write pointer with swaps.  
**Idea.** Swap every non-zero to `nums[write]`. Swapping (instead of copying) leaves the zeros behind the write pointer automatically.  
**Cost.** O(n) time · O(1) space

In [ ]:
def move_zeroes(nums: list[int]) -> None:
    """Stable move of zeros to the end."""
    write = 0
    for i, x in enumerate(nums):
        if x != 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1


a = [0, 1, 0, 3, 12]
move_zeroes(a)
assert a == [1, 3, 12, 0, 0]
print(a)

### 2.2.6 [189. Rotate Array](https://leetcode.com/problems/rotate-array/) — ★★ · Medium

**Problem.** Rotate right by `k` steps in place.  
**Signal.** Rotation in place → three reversals.  
**Idea.** Rotating right by `k` = reverse all, reverse the first `k`, reverse the rest. Take `k %= n` first.  
**Cost.** O(n) time · O(1) space

In [ ]:
def rotate(nums: list[int], k: int) -> None:
    """Rotate nums right by k in place."""

    def reverse(lo: int, hi: int) -> None:
        while lo < hi:
            nums[lo], nums[hi] = nums[hi], nums[lo]
            lo, hi = lo + 1, hi - 1

    n = len(nums)
    k %= n
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)


a = [1, 2, 3, 4, 5, 6, 7]
rotate(a, 3)
assert a == [5, 6, 7, 1, 2, 3, 4]
b = [-1, -100, 3, 99]
rotate(b, 2)
assert b == [3, 99, -1, -100]
print(a)

### 2.2.7 [75. Sort Colors](https://leetcode.com/problems/sort-colors/) — ★★ · Medium

**Problem.** Sort an array of 0s, 1s and 2s in one pass, in place.  
**Signal.** Three-way partition → Dutch national flag (three pointers).  
**Idea.** Invariant: `[0, low)` are 0s, `[low, mid)` are 1s, `(high, end]` are 2s. Look at `nums[mid]`: 0 → swap to `low`, advance both; 1 → advance `mid`; 2 → swap to `high`, shrink `high` but *don't* advance `mid` (the swapped-in value is unseen).  
**Cost.** O(n) time · O(1) space

In [ ]:
def sort_colors(nums: list[int]) -> None:
    """In-place one-pass sort of 0/1/2 values."""
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1


a = [2, 0, 2, 1, 1, 0]
sort_colors(a)
assert a == [0, 0, 1, 1, 2, 2]
print(a)

### 2.2.8 [31. Next Permutation](https://leetcode.com/problems/next-permutation/) — ★★ · Medium

**Problem.** Rearrange into the next lexicographically larger permutation (wrap to smallest), in place.  
**Signal.** Classic three-step algorithm: find pivot, swap with successor, reverse suffix.  
**Idea.** From the right, find the first `i` with `nums[i] < nums[i+1]` — the suffix after `i` is non-increasing, i.e. already the largest arrangement. Swap `nums[i]` with the smallest suffix value larger than it (the rightmost one > `nums[i]`), then reverse the suffix to make it smallest. No pivot → reverse everything.  
**Cost.** O(n) time · O(1) space

In [ ]:
def next_permutation(nums: list[int]) -> None:
    """Advance nums to its next permutation in place."""
    i = len(nums) - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = len(nums) - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1 :] = reversed(nums[i + 1 :])


for before, after in [
    ([1, 2, 3], [1, 3, 2]),
    ([3, 2, 1], [1, 2, 3]),
    ([1, 1, 5], [1, 5, 1]),
]:
    next_permutation(before)
    assert before == after
a = [1, 3, 5, 4, 2]
next_permutation(a)
print(a)

---
# 3) Two Pointers

Two indices that move monotonically replace a nested loop. Two shapes cover almost all
questions:

| Shape | Pointers | Why it is safe to discard |
|---|---|---|
| Opposite ends | `lo = 0`, `hi = n-1`, move inward | sorted order tells which end cannot be part of an answer |
| Same direction | `slow`, `fast` | `slow` marks the processed prefix (also used for in-place filtering above) |

## 3.1 Opposite-end pointers

### 3.1.1 [125. Valid Palindrome](https://leetcode.com/problems/valid-palindrome/) — ★★★ · Easy

**Problem.** After lower-casing and removing non-alphanumerics, is `s` a palindrome?  
**Signal.** Palindrome check → pointers from both ends.  
**Idea.** Skip non-alphanumeric characters on each side, compare lower-cased characters, move inward. No need to build the cleaned string.  
**Cost.** O(n) time · O(1) space

In [ ]:
def is_palindrome(s: str) -> bool:
    """Alphanumeric, case-insensitive palindrome test."""
    lo, hi = 0, len(s) - 1
    while lo < hi:
        if not s[lo].isalnum():
            lo += 1
        elif not s[hi].isalnum():
            hi -= 1
        elif s[lo].lower() != s[hi].lower():
            return False
        else:
            lo, hi = lo + 1, hi - 1
    return True


assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car")
assert is_palindrome(" ")
print(is_palindrome("A man, a plan, a canal: Panama"))

### 3.1.2 [167. Two Sum II - Input Array Is Sorted](https://leetcode.com/problems/two-sum-ii-input-array-is-sorted/) — ★★ · Medium

**Problem.** Sorted array; return 1-based indices of two numbers summing to `target`, O(1) space.  
**Signal.** Sorted + pair sum → opposite pointers.  
**Idea.** If `a[lo] + a[hi]` is too small, `a[lo]` cannot pair with anything (the largest partner already failed) → `lo += 1`. Too large → `hi -= 1`. Each step discards one element.  
**Cost.** O(n) time · O(1) space

In [ ]:
def two_sum_sorted(numbers: list[int], target: int) -> list[int]:
    """1-based indices of the pair summing to target."""
    lo, hi = 0, len(numbers) - 1
    while lo < hi:
        total = numbers[lo] + numbers[hi]
        if total == target:
            return [lo + 1, hi + 1]
        if total < target:
            lo += 1
        else:
            hi -= 1
    return []


assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
assert two_sum_sorted([2, 3, 4], 6) == [1, 3]
assert two_sum_sorted([-1, 0], -1) == [1, 2]
print(two_sum_sorted([2, 7, 11, 15], 9))

### 3.1.3 [15. 3Sum](https://leetcode.com/problems/3sum/) — ★★★ · Medium

**Problem.** All unique triplets summing to 0.  
**Signal.** Triplet sum → sort, fix one, two-pointer the rest.  
**Idea.** Sort. For each `i` (skipping repeated values of `nums[i]`), run Two Sum II on the suffix with target `-nums[i]`. After a hit, move both pointers and skip duplicates on the left. Stop early when `nums[i] > 0`.  
**Cost.** O(n²) time · O(1) extra space (sorting aside)  
**Trap.** Duplicates must be skipped at *both* levels to keep triplets unique.

In [ ]:
def three_sum(nums: list[int]) -> list[list[int]]:
    """Unique triplets with zero sum."""
    nums = sorted(nums)
    out: list[list[int]] = []
    for i in range(len(nums) - 2):
        if nums[i] > 0:
            break
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if total < 0:
                lo += 1
            elif total > 0:
                hi -= 1
            else:
                out.append([nums[i], nums[lo], nums[hi]])
                lo += 1
                hi -= 1
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1
    return out


assert three_sum([-1, 0, 1, 2, -1, -4]) == [[-1, -1, 2], [-1, 0, 1]]
assert three_sum([0, 1, 1]) == []
assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]
print(three_sum([-1, 0, 1, 2, -1, -4]))

### 3.1.4 [11. Container With Most Water](https://leetcode.com/problems/container-with-most-water/) — ★★★ · Medium

**Problem.** Choose two lines maximising `min(h[i], h[j]) * (j - i)`.  
**Signal.** Pair maximisation on an array → opposite pointers, move the limiting side.  
**Idea.** Start with the widest container. The shorter line limits the area; moving the taller line can only shrink width without raising the limit, so always move the shorter one.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_area(height: list[int]) -> int:
    """Largest water area between two lines."""
    lo, hi, best = 0, len(height) - 1, 0
    while lo < hi:
        best = max(best, min(height[lo], height[hi]) * (hi - lo))
        if height[lo] < height[hi]:
            lo += 1
        else:
            hi -= 1
    return best


assert max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert max_area([1, 1]) == 1
print(max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))

### 3.1.5 [42. Trapping Rain Water](https://leetcode.com/problems/trapping-rain-water/) — ★★ · Hard

**Problem.** Total water trapped between bars of the given heights.  
**Signal.** Water at i = `min(max_left, max_right) − h[i]` → two pointers with running maxima.  
**Idea.** Keep `left_max` and `right_max`. Whichever side has the smaller max is the bottleneck for its pointer — the other side is guaranteed at least as high — so settle that pointer: add `side_max − h` and move it inward.  
**Cost.** O(n) time · O(1) space

In [ ]:
def trap(height: list[int]) -> int:
    """Units of rain water trapped."""
    lo, hi = 0, len(height) - 1
    left_max = right_max = water = 0
    while lo < hi:
        if height[lo] < height[hi]:
            left_max = max(left_max, height[lo])
            water += left_max - height[lo]
            lo += 1
        else:
            right_max = max(right_max, height[hi])
            water += right_max - height[hi]
            hi -= 1
    return water


assert trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert trap([4, 2, 0, 3, 2, 5]) == 9
print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))

### 3.1.6 [680. Valid Palindrome II](https://leetcode.com/problems/valid-palindrome-ii/) — ★★ · Easy

**Problem.** Can `s` become a palindrome by deleting at most one character?  
**Signal.** Palindrome with one mismatch allowed → on the first mismatch, try both deletions.  
**Idea.** Move pointers inward while characters match. At the first mismatch, the answer is whether `s[lo+1..hi]` or `s[lo..hi-1]` is a palindrome. Only one branch point, so linear.  
**Cost.** O(n) time · O(1) space

In [ ]:
def valid_palindrome_ii(s: str) -> bool:
    """True if deleting at most one char yields a palindrome."""

    def is_pal(lo: int, hi: int) -> bool:
        while lo < hi:
            if s[lo] != s[hi]:
                return False
            lo, hi = lo + 1, hi - 1
        return True

    lo, hi = 0, len(s) - 1
    while lo < hi:
        if s[lo] != s[hi]:
            return is_pal(lo + 1, hi) or is_pal(lo, hi - 1)
        lo, hi = lo + 1, hi - 1
    return True


assert valid_palindrome_ii("aba")
assert valid_palindrome_ii("abca")
assert not valid_palindrome_ii("abc")
print(valid_palindrome_ii("abca"))

### 3.1.7 [977. Squares of a Sorted Array](https://leetcode.com/problems/squares-of-a-sorted-array/) — ★★ · Easy

**Problem.** Sorted array (may include negatives); return sorted squares in O(n).  
**Signal.** Largest squares sit at the two ends → opposite pointers, fill output from the back.  
**Idea.** Compare `|a[lo]|` and `|a[hi]|`; the larger square goes to the last free output slot.  
**Cost.** O(n) time · O(n) output

In [ ]:
def sorted_squares(nums: list[int]) -> list[int]:
    """Sorted squares of a sorted array."""
    n = len(nums)
    out = [0] * n
    lo, hi = 0, n - 1
    for k in range(n - 1, -1, -1):
        if abs(nums[lo]) > abs(nums[hi]):
            out[k] = nums[lo] ** 2
            lo += 1
        else:
            out[k] = nums[hi] ** 2
            hi -= 1
    return out


assert sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]
assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121]
print(sorted_squares([-4, -1, 0, 3, 10]))

### 3.1.8 [16. 3Sum Closest](https://leetcode.com/problems/3sum-closest/) — ★★ · Medium

**Problem.** Sum of the triplet closest to `target`.  
**Signal.** Same as 3Sum, but track the best distance instead of exact hits.  
**Idea.** Sort; for each `i`, two pointers on the suffix. Update the best sum whenever `|total − target|` improves, and move the pointer that brings `total` toward `target`.  
**Cost.** O(n²) time · O(1) space

In [ ]:
def three_sum_closest(nums: list[int], target: int) -> int:
    """Triplet sum nearest to target."""
    nums = sorted(nums)
    best = nums[0] + nums[1] + nums[2]
    for i in range(len(nums) - 2):
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if abs(total - target) < abs(best - target):
                best = total
            if total < target:
                lo += 1
            elif total > target:
                hi -= 1
            else:
                return total
    return best


assert three_sum_closest([-1, 2, 1, -4], 1) == 2
assert three_sum_closest([0, 0, 0], 1) == 0
print(three_sum_closest([-1, 2, 1, -4], 1))

### 3.1.9 [18. 4Sum](https://leetcode.com/problems/4sum/) — ★ · Medium

**Problem.** All unique quadruplets summing to `target`.  
**Signal.** k-Sum → recursion that peels one element per level down to Two Sum II.  
**Idea.** Sort. `k_sum(start, k, target)` loops over a first element (skipping duplicates) and recurses with `k-1`; at `k == 2` use two pointers. The same function solves any k.  
**Cost.** O(n^{k-1}) time — O(n³) for 4Sum

In [ ]:
def four_sum(nums: list[int], target: int) -> list[list[int]]:
    """Unique quadruplets summing to target."""
    nums = sorted(nums)

    def k_sum(start: int, k: int, goal: int) -> list[list[int]]:
        out: list[list[int]] = []
        if k == 2:
            lo, hi = start, len(nums) - 1
            while lo < hi:
                total = nums[lo] + nums[hi]
                if total < goal or (lo > start and nums[lo] == nums[lo - 1]):
                    lo += 1
                elif total > goal:
                    hi -= 1
                else:
                    out.append([nums[lo], nums[hi]])
                    lo += 1
                    hi -= 1
            return out
        for i in range(start, len(nums) - k + 1):
            if i > start and nums[i] == nums[i - 1]:
                continue
            for rest in k_sum(i + 1, k - 1, goal - nums[i]):
                out.append([nums[i]] + rest)
        return out

    return k_sum(0, 4, target)


assert four_sum([1, 0, -1, 0, -2, 2], 0) == [
    [-2, -1, 1, 2],
    [-2, 0, 0, 2],
    [-1, 0, 0, 1],
]
assert four_sum([2, 2, 2, 2, 2], 8) == [[2, 2, 2, 2]]
print(four_sum([1, 0, -1, 0, -2, 2], 0))

### 3.1.10 [344. Reverse String](https://leetcode.com/problems/reverse-string/) — ★ · Easy

**Problem.** Reverse a list of characters in place.  
**Signal.** In-place reversal → swap ends, move inward.  
**Idea.** Swap `s[lo]` and `s[hi]` until they meet. This swap loop is the building block of Rotate Array, Next Permutation and Reverse Words.  
**Cost.** O(n) time · O(1) space

In [ ]:
def reverse_string(s: list[str]) -> None:
    """Reverse s in place."""
    lo, hi = 0, len(s) - 1
    while lo < hi:
        s[lo], s[hi] = s[hi], s[lo]
        lo, hi = lo + 1, hi - 1


chars = list("hello")
reverse_string(chars)
assert chars == list("olleh")
print(chars)

### 3.1.11 [881. Boats to Save People](https://leetcode.com/problems/boats-to-save-people/) — ★ · Medium

**Problem.** Each boat holds at most 2 people and weight `limit`; minimum number of boats.  
**Signal.** Pairing heaviest with lightest → sort + opposite pointers (greedy).  
**Idea.** The heaviest person must take a boat. If the lightest person fits with them, they share it (pairing the lightest wastes the least capacity); otherwise the heaviest goes alone.  
**Cost.** O(n log n) time · O(1) extra space

In [ ]:
def num_rescue_boats(people: list[int], limit: int) -> int:
    """Fewest two-seat boats needed."""
    people = sorted(people)
    lo, hi, boats = 0, len(people) - 1, 0
    while lo <= hi:
        if people[lo] + people[hi] <= limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats


assert num_rescue_boats([1, 2], 3) == 1
assert num_rescue_boats([3, 2, 2, 1], 3) == 3
assert num_rescue_boats([3, 5, 3, 4], 5) == 4
print(num_rescue_boats([3, 2, 2, 1], 3))

## 3.2 Same-direction pointers

### 3.2.1 [392. Is Subsequence](https://leetcode.com/problems/is-subsequence/) — ★★ · Easy

**Problem.** Is `s` a subsequence of `t`?  
**Signal.** Match in order, skipping allowed → one pointer per string.  
**Idea.** Walk through `t`; advance the `s` pointer whenever characters match. Greedy matching is optimal because matching earlier never hurts later matches.  
**Cost.** O(|t|) time · O(1) space

In [ ]:
def is_subsequence(s: str, t: str) -> bool:
    """True if s can be obtained by deleting chars from t."""
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)


assert is_subsequence("abc", "ahbgdc")
assert not is_subsequence("axc", "ahbgdc")
assert is_subsequence("", "abc")
print(is_subsequence("abc", "ahbgdc"))

### 3.2.2 [844. Backspace String Compare](https://leetcode.com/problems/backspace-string-compare/) — ★★ · Easy

**Problem.** `#` is a backspace; are `s` and `t` equal after typing? O(1) space.  
**Signal.** Backspace affects characters to the *left* → scan from the right with a skip counter.  
**Idea.** Walk each string from the end, counting pending backspaces and skipping that many characters; compare the next surviving characters of both strings.  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def backspace_compare(s: str, t: str) -> bool:
    """Equality of strings after applying '#' backspaces."""

    def survivors(text: str):
        skip = 0
        for ch in reversed(text):
            if ch == "#":
                skip += 1
            elif skip:
                skip -= 1
            else:
                yield ch

    return list(survivors(s)) == list(survivors(t))


assert backspace_compare("ab#c", "ad#c")
assert backspace_compare("ab##", "c#d#")
assert not backspace_compare("a#c", "b")
print(backspace_compare("ab#c", "ad#c"))

### 3.2.3 [1768. Merge Strings Alternately](https://leetcode.com/problems/merge-strings-alternately/) — ★ · Easy

**Problem.** Merge two strings by alternating characters; append the leftover tail.  
**Signal.** Walk both strings with one index.  
**Idea.** For `i` in range of the longer length, append `a[i]` and `b[i]` if they exist.  
**Cost.** O(m + n) time

In [ ]:
def merge_alternately(word1: str, word2: str) -> str:
    """Interleave two strings."""
    out = []
    for i in range(max(len(word1), len(word2))):
        out.append(word1[i : i + 1])
        out.append(word2[i : i + 1])
    return "".join(out)


assert merge_alternately("abc", "pqr") == "apbqcr"
assert merge_alternately("ab", "pqrs") == "apbqrs"
print(merge_alternately("abcd", "pq"))

### 3.2.4 [151. Reverse Words in a String](https://leetcode.com/problems/reverse-words-in-a-string/) — ★★ · Medium

**Problem.** Reverse the order of words; collapse extra spaces.  
**Signal.** In Python `split()` handles spaces; in-place version = reverse whole, then each word.  
**Idea.** `" ".join(reversed(s.split()))`. In an interview with mutable char arrays: reverse the whole string, then reverse each word back, then compact spaces with a write pointer.  
**Cost.** O(n) time

In [ ]:
def reverse_words(s: str) -> str:
    """Words of s in reverse order, single-spaced."""
    return " ".join(reversed(s.split()))


assert reverse_words("the sky is blue") == "blue is sky the"
assert reverse_words("  hello world  ") == "world hello"
assert reverse_words("a good   example") == "example good a"
print(reverse_words("the sky is blue"))

### 3.2.5 [28. Find the Index of the First Occurrence in a String](https://leetcode.com/problems/find-the-index-of-the-first-occurrence-in-a-string/) — ★★ · Easy

**Problem.** Index of the first occurrence of `needle` in `haystack`, or −1.  
**Signal.** Substring search → sliding comparison (KMP for guaranteed linear time).  
**Idea.** KMP precomputes `lps[i]`, the longest proper prefix of `needle[:i+1]` that is also a suffix. On a mismatch, instead of restarting, jump the needle pointer to `lps[j-1]` — the part already matched is reused, so the haystack pointer never moves back.  
**Cost.** O(m + n) time · O(n) space

In [ ]:
def str_str(haystack: str, needle: str) -> int:
    """First index of needle in haystack (KMP)."""
    lps = [0] * len(needle)
    j = 0
    for i in range(1, len(needle)):
        while j and needle[i] != needle[j]:
            j = lps[j - 1]
        if needle[i] == needle[j]:
            j += 1
        lps[i] = j
    j = 0
    for i, ch in enumerate(haystack):
        while j and ch != needle[j]:
            j = lps[j - 1]
        if ch == needle[j]:
            j += 1
        if j == len(needle):
            return i - j + 1
    return -1


assert str_str("sadbutsad", "sad") == 0
assert str_str("leetcode", "leeto") == -1
assert str_str("aabaaabaaac", "aabaaac") == 4
print(str_str("hello", "ll"))

---
# 4) Sliding Window

A window `[left, right]` over a contiguous block. `right` grows the window one step at a
time; `left` shrinks it when the window breaks a constraint. Each index enters and leaves
once, so the scan is $O(n)$.

```python
left = 0
for right, x in enumerate(arr):
    add(x)                          # extend window
    while window_is_invalid():      # shrink until valid
        remove(arr[left]); left += 1
    best = max(best, right - left + 1)   # longest valid window
```

For *shortest* valid window, update the answer *inside* the while-loop instead. The
technique needs **monotonicity**: if a window is invalid, every larger window containing
it is invalid too. With negative numbers this breaks and you need prefix sums instead.

## 4.1 Variable-size windows

### 4.1.1 [121. Best Time to Buy and Sell Stock](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) — ★★★ · Easy

**Problem.** One buy then one later sell; maximum profit (0 if none).  
**Signal.** Best pair `i < j` maximising `p[j] − p[i]` → track the running minimum.  
**Idea.** For each day as the sell day, the best buy day is the cheapest day so far. Keep `min_price` and update the best profit.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_profit(prices: list[int]) -> int:
    """Best single-trade profit."""
    min_price, best = float("inf"), 0
    for p in prices:
        min_price = min(min_price, p)
        best = max(best, p - min_price)
    return best


assert max_profit([7, 1, 5, 3, 6, 4]) == 5
assert max_profit([7, 6, 4, 3, 1]) == 0
print(max_profit([7, 1, 5, 3, 6, 4]))

### 4.1.2 [3. Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/) — ★★★ · Medium

**Problem.** Length of the longest substring with all distinct characters.  
**Signal.** Longest contiguous block with a property → sliding window.  
**Idea.** Store each character's last index. When `s[right]` was seen inside the current window, jump `left` past that earlier occurrence (no need to shrink one step at a time).  
**Cost.** O(n) time · O(alphabet) space  
**Trap.** Only jump `left` forward — use `max(left, last[ch] + 1)`.

In [ ]:
def length_of_longest_substring(s: str) -> int:
    """Longest substring without repeated characters."""
    last: dict[str, int] = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last:
            left = max(left, last[ch] + 1)
        last[ch] = right
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("abba") == 2
print(length_of_longest_substring("abcabcbb"))

### 4.1.3 [424. Longest Repeating Character Replacement](https://leetcode.com/problems/longest-repeating-character-replacement/) — ★★★ · Medium

**Problem.** Replace at most `k` characters; longest substring of one repeated letter.  
**Signal.** Window is valid while `size − (count of most frequent char) ≤ k`.  
**Idea.** Grow the window and track `max_freq`, the highest single-letter count ever seen in a window. If `size − max_freq > k`, slide `left` by one. `max_freq` never has to decrease: the answer only improves when a window beats the old `max_freq`, so a stale value can only keep the window size unchanged, never produce a wrong larger answer.  
**Cost.** O(n) time · O(26) space

In [ ]:
def character_replacement(s: str, k: int) -> int:
    """Longest same-letter substring after at most k replacements."""
    count: Counter[str] = Counter()
    left = max_freq = 0
    for right, ch in enumerate(s):
        count[ch] += 1
        max_freq = max(max_freq, count[ch])
        if right - left + 1 - max_freq > k:
            count[s[left]] -= 1
            left += 1
    return len(s) - left


assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
print(character_replacement("AABABBA", 1))

### 4.1.4 [76. Minimum Window Substring](https://leetcode.com/problems/minimum-window-substring/) — ★★★ · Hard

**Problem.** Smallest substring of `s` that contains every character of `t` (with multiplicity).  
**Signal.** Shortest window covering a requirement → expand until valid, shrink while valid.  
**Idea.** `need` counts characters of `t`; `missing` = how many required characters are still uncovered. Expand `right`, decrementing `need`. When `missing == 0`, shrink `left` while the window stays valid, recording the best window each time.  
**Cost.** O(|s| + |t|) time · O(alphabet) space

In [ ]:
def min_window(s: str, t: str) -> str:
    """Shortest substring of s containing all of t."""
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (0, float("inf"))
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:
            if right - left < best[1] - best[0]:
                best = (left, right)
            need[s[left]] += 1
            if need[s[left]] > 0:
                missing += 1
            left += 1
    return "" if best[1] == float("inf") else s[best[0] : best[1] + 1]


assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
print(min_window("ADOBECODEBANC", "ABC"))

### 4.1.5 [209. Minimum Size Subarray Sum](https://leetcode.com/problems/minimum-size-subarray-sum/) — ★★ · Medium

**Problem.** Positive integers; length of the shortest subarray with sum ≥ `target` (0 if none).  
**Signal.** Positive values → window sum is monotone → shortest-window template.  
**Idea.** Add `nums[right]`; while the sum is ≥ target, record the length and remove `nums[left]`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def min_sub_array_len(target: int, nums: list[int]) -> int:
    """Shortest subarray length with sum >= target."""
    left = total = 0
    best = len(nums) + 1
    for right, x in enumerate(nums):
        total += x
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return best if best <= len(nums) else 0


assert min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2
assert min_sub_array_len(4, [1, 4, 4]) == 1
assert min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0
print(min_sub_array_len(7, [2, 3, 1, 2, 4, 3]))

### 4.1.6 [1004. Max Consecutive Ones III](https://leetcode.com/problems/max-consecutive-ones-iii/) — ★ · Medium

**Problem.** Binary array; flip at most `k` zeros; longest run of 1s.  
**Signal.** "At most k bad elements in the window" → longest-window template counting zeros.  
**Idea.** Count zeros inside the window; while more than `k`, shrink from the left.  
**Cost.** O(n) time · O(1) space

In [ ]:
def longest_ones(nums: list[int], k: int) -> int:
    """Longest 1-run after flipping at most k zeros."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        while zeros > k:
            zeros -= nums[left] == 0
            left += 1
        best = max(best, right - left + 1)
    return best


assert longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2) == 6
assert longest_ones([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3) == 10
print(longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))

### 4.1.7 [1493. Longest Subarray of 1's After Deleting One Element](https://leetcode.com/problems/longest-subarray-of-1s-after-deleting-one-element/) — ★ · Medium

**Problem.** Delete exactly one element; longest subarray of only 1s.  
**Signal.** Same as Max Consecutive Ones III with `k = 1`, minus the deleted element.  
**Idea.** Longest window with at most one zero, then subtract 1 for the mandatory deletion (also correct for an all-ones array, where a 1 must be deleted).  
**Cost.** O(n) time · O(1) space

In [ ]:
def longest_subarray_after_delete(nums: list[int]) -> int:
    """Longest all-ones subarray after deleting one element."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1:
            zeros -= nums[left] == 0
            left += 1
        best = max(best, right - left)
    return best


assert longest_subarray_after_delete([1, 1, 0, 1]) == 3
assert longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]) == 5
assert longest_subarray_after_delete([1, 1, 1]) == 2
print(longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]))

### 4.1.8 [904. Fruit Into Baskets](https://leetcode.com/problems/fruit-into-baskets/) — ★ · Medium

**Problem.** Longest subarray containing at most 2 distinct values.  
**Signal.** "At most k distinct" → window with a count map; shrink while map has > k keys.  
**Idea.** Add `fruits[right]` to the counts; while more than two kinds, remove `fruits[left]` (deleting zero-count keys).  
**Cost.** O(n) time · O(1) space

In [ ]:
def total_fruit(fruits: list[int]) -> int:
    """Longest subarray with at most two distinct values."""
    count: Counter[int] = Counter()
    left = best = 0
    for right, f in enumerate(fruits):
        count[f] += 1
        while len(count) > 2:
            count[fruits[left]] -= 1
            if count[fruits[left]] == 0:
                del count[fruits[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert total_fruit([1, 2, 1]) == 3
assert total_fruit([0, 1, 2, 2]) == 3
assert total_fruit([1, 2, 3, 2, 2]) == 4
print(total_fruit([1, 2, 3, 2, 2]))

### 4.1.9 [340. Longest Substring with At Most K Distinct Characters](https://leetcode.com/problems/longest-substring-with-at-most-k-distinct-characters/) — ★ · Medium · Premium

**Problem.** Longest substring with at most `k` distinct characters.  
**Signal.** Fruit Into Baskets with a general `k`.  
**Idea.** Identical template; the window is valid while `len(count) <= k`.  
**Cost.** O(n) time · O(k) space

In [ ]:
def length_of_longest_substring_k_distinct(s: str, k: int) -> int:
    """Longest substring with at most k distinct characters."""
    count: Counter[str] = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        count[ch] += 1
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring_k_distinct("eceba", 2) == 3
assert length_of_longest_substring_k_distinct("aa", 1) == 2
print(length_of_longest_substring_k_distinct("eceba", 2))

### 4.1.10 [992. Subarrays with K Different Integers](https://leetcode.com/problems/subarrays-with-k-different-integers/) — ★ · Hard

**Problem.** Count subarrays with exactly `k` distinct values.  
**Signal.** "Exactly k" is not monotone, but "at most k" is → `exact(k) = at_most(k) − at_most(k−1)`.  
**Idea.** With the longest-window template, every valid window ending at `right` contributes `right − left + 1` subarrays (all its suffixes). Count those for at most `k` and subtract the count for at most `k − 1`.  
**Cost.** O(n) time · O(k) space

In [ ]:
def subarrays_with_k_distinct(nums: list[int], k: int) -> int:
    """Number of subarrays with exactly k distinct values."""

    def at_most(limit: int) -> int:
        count: Counter[int] = Counter()
        left = total = 0
        for right, x in enumerate(nums):
            count[x] += 1
            while len(count) > limit:
                count[nums[left]] -= 1
                if count[nums[left]] == 0:
                    del count[nums[left]]
                left += 1
            total += right - left + 1
        return total

    return at_most(k) - at_most(k - 1)


assert subarrays_with_k_distinct([1, 2, 1, 2, 3], 2) == 7
assert subarrays_with_k_distinct([1, 2, 1, 3, 4], 3) == 3
print(subarrays_with_k_distinct([1, 2, 1, 2, 3], 2))

### 4.1.11 [1838. Frequency of the Most Frequent Element](https://leetcode.com/problems/frequency-of-the-most-frequent-element/) — ★ · Medium

**Problem.** Up to `k` increments by 1 in total; maximise the count of one value.  
**Signal.** After sorting, make a window all equal to its largest element; cost is `max × size − sum`.  
**Idea.** Sort. For window `[left, right]`, raising everything to `nums[right]` costs `nums[right] * size − window_sum`. Shrink while the cost exceeds `k`.  
**Cost.** O(n log n) time · O(1) space

In [ ]:
def max_frequency(nums: list[int], k: int) -> int:
    """Highest achievable frequency with k unit increments."""
    nums = sorted(nums)
    left = total = best = 0
    for right, x in enumerate(nums):
        total += x
        while x * (right - left + 1) - total > k:
            total -= nums[left]
            left += 1
        best = max(best, right - left + 1)
    return best


assert max_frequency([1, 2, 4], 5) == 3
assert max_frequency([1, 4, 8, 13], 5) == 2
assert max_frequency([3, 9, 6], 2) == 1
print(max_frequency([1, 4, 8, 13], 5))

### 4.1.12 [1658. Minimum Operations to Reduce X to Zero](https://leetcode.com/problems/minimum-operations-to-reduce-x-to-zero/) — ★ · Medium

**Problem.** Remove elements from either end so the removed sum is exactly `x`; fewest removals.  
**Signal.** Removing from both ends = keeping a middle block → longest subarray with sum `total − x`.  
**Idea.** Invert the problem: find the longest contiguous window with sum `total − x` (all values positive, so a sliding window works). Answer `n − longest`, or −1.  
**Cost.** O(n) time · O(1) space

In [ ]:
def min_operations(nums: list[int], x: int) -> int:
    """Fewest end-removals summing to x."""
    goal = sum(nums) - x
    if goal < 0:
        return -1
    left = window = 0
    longest = -1
    for right, v in enumerate(nums):
        window += v
        while window > goal:
            window -= nums[left]
            left += 1
        if window == goal:
            longest = max(longest, right - left + 1)
    return -1 if longest == -1 else len(nums) - longest


assert min_operations([1, 1, 4, 2, 3], 5) == 2
assert min_operations([5, 6, 7, 8, 9], 4) == -1
assert min_operations([3, 2, 20, 1, 1, 3], 10) == 5
print(min_operations([3, 2, 20, 1, 1, 3], 10))

## 4.2 Fixed-size windows

For a window of fixed length `k`, add `arr[right]` and remove `arr[right - k]` on every
step after the first `k`.

### 4.2.1 [567. Permutation in String](https://leetcode.com/problems/permutation-in-string/) — ★★ · Medium

**Problem.** Does `s2` contain a permutation of `s1` as a substring?  
**Signal.** Anagram of a fixed-length pattern → fixed window of size `len(s1)` with counts.  
**Idea.** Keep character counts of the current window and compare with `s1`'s counts. Sliding one step changes two counts. (A `matches` counter over 26 letters makes each comparison O(1); comparing arrays of 26 is also O(1).)  
**Cost.** O(n) time · O(26) space

In [ ]:
def check_inclusion(s1: str, s2: str) -> bool:
    """True if some substring of s2 is an anagram of s1."""
    k = len(s1)
    target = Counter(s1)
    window = Counter(s2[:k])
    if window == target:
        return True
    for right in range(k, len(s2)):
        window[s2[right]] += 1
        window[s2[right - k]] -= 1
        if window == target:
            return True
    return False


assert check_inclusion("ab", "eidbaooo")
assert not check_inclusion("ab", "eidboaoo")
print(check_inclusion("ab", "eidbaooo"))

### 4.2.2 [438. Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/) — ★★ · Medium

**Problem.** All start indices in `s` of anagrams of `p`.  
**Signal.** Permutation in String, collecting every match instead of stopping.  
**Idea.** Fixed window of length `len(p)` with counts; record `right − k + 1` whenever counts equal.  
**Cost.** O(n) time · O(26) space  
**Trap.** `Counter` equality treats a zero count like a missing key (Python 3.10+), which is what we want here.

In [ ]:
def find_anagrams(s: str, p: str) -> list[int]:
    """Start indices of p's anagrams in s."""
    k = len(p)
    target, window = Counter(p), Counter()
    out = []
    for right, ch in enumerate(s):
        window[ch] += 1
        if right >= k:
            window[s[right - k]] -= 1
        if window == target:
            out.append(right - k + 1)
    return out


assert find_anagrams("cbaebabacd", "abc") == [0, 6]
assert find_anagrams("abab", "ab") == [0, 1, 2]
print(find_anagrams("cbaebabacd", "abc"))

### 4.2.3 [643. Maximum Average Subarray I](https://leetcode.com/problems/maximum-average-subarray-i/) — ★ · Easy

**Problem.** Maximum average of a contiguous subarray of length `k`.  
**Signal.** Fixed window → running sum.  
**Idea.** Sum the first `k`, then slide: add the new element, subtract the one leaving.  
**Cost.** O(n) time · O(1) space

In [ ]:
def find_max_average(nums: list[int], k: int) -> float:
    """Best average over windows of length k."""
    window = best = sum(nums[:k])
    for right in range(k, len(nums)):
        window += nums[right] - nums[right - k]
        best = max(best, window)
    return best / k


assert find_max_average([1, 12, -5, -6, 50, 3], 4) == 12.75
assert find_max_average([5], 1) == 5.0
print(find_max_average([1, 12, -5, -6, 50, 3], 4))

### 4.2.4 [1423. Maximum Points You Can Obtain from Cards](https://leetcode.com/problems/maximum-points-you-can-obtain-from-cards/) — ★ · Medium

**Problem.** Take exactly `k` cards from the two ends; maximise the sum.  
**Signal.** Taking from ends = leaving a contiguous middle of length `n − k`.  
**Idea.** Minimise the sum of a fixed window of length `n − k`; answer is `total − min_window`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_score(card_points: list[int], k: int) -> int:
    """Best sum of k cards taken from the ends."""
    n = len(card_points)
    size = n - k
    window = best = sum(card_points[:size])
    for right in range(size, n):
        window += card_points[right] - card_points[right - size]
        best = min(best, window)
    return sum(card_points) - best


assert max_score([1, 2, 3, 4, 5, 6, 1], 3) == 12
assert max_score([2, 2, 2], 2) == 4
assert max_score([9, 7, 7, 9, 7, 7, 9], 7) == 55
print(max_score([1, 2, 3, 4, 5, 6, 1], 3))

### 4.2.5 [239. Sliding Window Maximum](https://leetcode.com/problems/sliding-window-maximum/) — ★★ · Hard

**Problem.** Maximum of every window of size `k`.  
**Signal.** Window max with O(1) amortised updates → monotonic decreasing deque of indices.  
**Idea.** Keep indices whose values decrease from front to back. Before pushing `i`, pop smaller values from the back — they can never be a max while `i` is in the window. Pop the front when it slides out. The front is always the current max.  
**Cost.** O(n) time (each index pushed and popped once) · O(k) space

In [ ]:
def max_sliding_window(nums: list[int], k: int) -> list[int]:
    """Maximum of each length-k window."""
    dq: deque[int] = deque()
    out = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] == i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
print(max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))

### 4.2.6 [30. Substring with Concatenation of All Words](https://leetcode.com/problems/substring-with-concatenation-of-all-words/) — ★★ · Hard

**Problem.** Start indices where a concatenation of all `words` (equal length `w`, any order) appears.  
**Signal.** Fixed word length → run a word-level sliding window for each of the `w` offsets.  
**Idea.** For each offset `0..w−1`, step through `s` in chunks of `w`. Maintain counts of words in the window; if a word is unknown, reset; if a word is over-used, shrink from the left. When the window holds exactly `len(words)` words, record `left`.  
**Cost.** O(n · w) time · O(m) space for m words

In [ ]:
def find_substring(s: str, words: list[str]) -> list[int]:
    """Starts of concatenations of all words."""
    w, m = len(words[0]), len(words)
    need = Counter(words)
    out = []
    for offset in range(w):
        left, window, used = offset, Counter(), 0
        for right in range(offset, len(s) - w + 1, w):
            word = s[right : right + w]
            if word not in need:
                left, window, used = right + w, Counter(), 0
                continue
            window[word] += 1
            used += 1
            while window[word] > need[word]:
                window[s[left : left + w]] -= 1
                left += w
                used -= 1
            if used == m:
                out.append(left)
    return sorted(out)


assert find_substring("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
assert (
    find_substring("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
)
assert find_substring("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]
print(find_substring("barfoothefoobarman", ["foo", "bar"]))

---
# 5) Prefix Sums

`prefix[i] = nums[0] + … + nums[i-1]`, so any range sum is `prefix[j] − prefix[i]`. The
powerful variant pairs it with a hash map: "number of subarrays with sum k" becomes "how
many earlier prefixes equal `current − k`?" — this works with negative numbers, where a
sliding window does not.

## 5.1 Prefix sum with hash map

### 5.1.1 [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/) — ★★ · Medium

**Problem.** Count contiguous subarrays summing to `k` (values may be negative).  
**Signal.** Negative values break sliding windows → prefix sum + hash map of prefix counts.  
**Idea.** A subarray ending at `i` sums to `k` iff some earlier prefix equals `prefix_i − k`. Keep a count of every prefix seen (start with `{0: 1}` for subarrays starting at index 0).  
**Cost.** O(n) time · O(n) space

In [ ]:
def subarray_sum(nums: list[int], k: int) -> int:
    """Count of subarrays with sum k."""
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix += x
        total += seen[prefix - k]
        seen[prefix] += 1
    return total


assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, 2, 3], 3) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print(subarray_sum([1, 1, 1], 2))

### 5.1.2 [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/) — ★★ · Medium

**Problem.** Binary array; longest subarray with equal numbers of 0s and 1s.  
**Signal.** Map 0 → −1; equal counts ⇔ subarray sum 0 ⇔ two equal prefix sums.  
**Idea.** Store the *first* index of each prefix sum. When a prefix repeats, the subarray between the two occurrences sums to 0; the earliest occurrence gives the longest one.  
**Cost.** O(n) time · O(n) space

In [ ]:
def find_max_length(nums: list[int]) -> int:
    """Longest balanced 0/1 subarray."""
    first = {0: -1}
    prefix = best = 0
    for i, x in enumerate(nums):
        prefix += 1 if x else -1
        if prefix in first:
            best = max(best, i - first[prefix])
        else:
            first[prefix] = i
    return best


assert find_max_length([0, 1]) == 2
assert find_max_length([0, 1, 0]) == 2
assert find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]) == 6
print(find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]))

### 5.1.3 [974. Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/) — ★ · Medium

**Problem.** Count subarrays whose sum is divisible by `k`.  
**Signal.** Divisible ⇔ two prefixes have the same remainder mod `k`.  
**Idea.** Count prefix remainders; each new prefix pairs with every earlier prefix of the same remainder. Python's `%` already returns non-negative remainders for negative sums.  
**Cost.** O(n) time · O(k) space

In [ ]:
def subarrays_div_by_k(nums: list[int], k: int) -> int:
    """Count subarrays with sum % k == 0."""
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix = (prefix + x) % k
        total += seen[prefix]
        seen[prefix] += 1
    return total


assert subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5) == 7
assert subarrays_div_by_k([5], 9) == 0
print(subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5))

### 5.1.4 [523. Continuous Subarray Sum](https://leetcode.com/problems/continuous-subarray-sum/) — ★ · Medium

**Problem.** Is there a subarray of length ≥ 2 whose sum is a multiple of `k`?  
**Signal.** Same remainder twice, at least two indices apart.  
**Idea.** Store the first index of each prefix remainder (with `{0: -1}`). When a remainder repeats at index `i` and `i − first ≥ 2`, return `True`.  
**Cost.** O(n) time · O(k) space

In [ ]:
def check_subarray_sum(nums: list[int], k: int) -> bool:
    """True if a length>=2 subarray sums to a multiple of k."""
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix = (prefix + x) % k
        if prefix in first:
            if i - first[prefix] >= 2:
                return True
        else:
            first[prefix] = i
    return False


assert check_subarray_sum([23, 2, 4, 6, 7], 6)
assert check_subarray_sum([23, 2, 6, 4, 7], 6)
assert not check_subarray_sum([23, 2, 6, 4, 7], 13)
print(check_subarray_sum([23, 2, 4, 6, 7], 6))

## 5.2 Range queries and difference arrays

### 5.2.1 [303. Range Sum Query - Immutable](https://leetcode.com/problems/range-sum-query-immutable/) — ★ · Easy

**Problem.** Answer many `sum(nums[left..right])` queries.  
**Signal.** Repeated range sums on a static array → prefix array.  
**Idea.** Precompute `prefix` of length `n + 1`; each query is `prefix[right+1] − prefix[left]`.  
**Cost.** O(n) build · O(1) per query

In [ ]:
class NumArray:
    """Static range-sum structure."""

    def __init__(self, nums: list[int]) -> None:
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def sum_range(self, left: int, right: int) -> int:
        """Sum of nums[left..right]."""
        return self.prefix[right + 1] - self.prefix[left]


arr = NumArray([-2, 0, 3, -5, 2, -1])
assert [arr.sum_range(0, 2), arr.sum_range(2, 5), arr.sum_range(0, 5)] == [1, -1, -3]
print(arr.sum_range(0, 5))

### 5.2.2 [304. Range Sum Query 2D - Immutable](https://leetcode.com/problems/range-sum-query-2d-immutable/) — ★ · Medium

**Problem.** Many queries for the sum of a sub-rectangle.  
**Signal.** 2D prefix sums with inclusion–exclusion.  
**Idea.** `P[r+1][c+1]` = sum of the rectangle from (0,0) to (r,c). A query is `P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]` (the last term was subtracted twice).  
**Cost.** O(mn) build · O(1) per query

In [ ]:
class NumMatrix:
    """Static 2D range-sum structure."""

    def __init__(self, matrix: list[list[int]]) -> None:
        rows, cols = len(matrix), len(matrix[0])
        self.p = [[0] * (cols + 1) for _ in range(rows + 1)]
        for r in range(rows):
            for c in range(cols):
                self.p[r + 1][c + 1] = (
                    matrix[r][c] + self.p[r][c + 1] + self.p[r + 1][c] - self.p[r][c]
                )

    def sum_region(self, r1: int, c1: int, r2: int, c2: int) -> int:
        """Sum of matrix[r1..r2][c1..c2]."""
        p = self.p
        return p[r2 + 1][c2 + 1] - p[r1][c2 + 1] - p[r2 + 1][c1] + p[r1][c1]


m = NumMatrix(
    [
        [3, 0, 1, 4, 2],
        [5, 6, 3, 2, 1],
        [1, 2, 0, 1, 5],
        [4, 1, 0, 1, 7],
        [1, 0, 3, 0, 5],
    ]
)
assert m.sum_region(2, 1, 4, 3) == 8
assert m.sum_region(1, 1, 2, 2) == 11
assert m.sum_region(1, 2, 2, 4) == 12
print(m.sum_region(2, 1, 4, 3))

### 5.2.3 [724. Find Pivot Index](https://leetcode.com/problems/find-pivot-index/) — ★ · Easy

**Problem.** Leftmost index where the sum to the left equals the sum to the right.  
**Signal.** Left sum + right sum = total − nums[i] → one running sum.  
**Idea.** Walk left to right keeping `left_sum`; index `i` is a pivot iff `left_sum == total − left_sum − nums[i]`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def pivot_index(nums: list[int]) -> int:
    """Leftmost balance index, or -1."""
    total, left_sum = sum(nums), 0
    for i, x in enumerate(nums):
        if left_sum == total - left_sum - x:
            return i
        left_sum += x
    return -1


assert pivot_index([1, 7, 3, 6, 5, 6]) == 3
assert pivot_index([1, 2, 3]) == -1
assert pivot_index([2, 1, -1]) == 0
print(pivot_index([1, 7, 3, 6, 5, 6]))

### 5.2.4 [1109. Corporate Flight Bookings](https://leetcode.com/problems/corporate-flight-bookings/) — ★ · Medium

**Problem.** Bookings `[first, last, seats]` add seats to a range of flights; return totals per flight.  
**Signal.** Many range-add updates, one final read → difference array.  
**Idea.** For each booking add `seats` at `first` and subtract at `last + 1`. A single prefix sum over the difference array produces every total. Turns O(n · bookings) into O(n + bookings).  
**Cost.** O(n + b) time · O(n) space

In [ ]:
def corp_flight_bookings(bookings: list[list[int]], n: int) -> list[int]:
    """Seat totals per flight after range bookings."""
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        diff[first - 1] += seats
        diff[last] -= seats
    out, running = [], 0
    for i in range(n):
        running += diff[i]
        out.append(running)
    return out


assert corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5) == [
    10,
    55,
    45,
    25,
    25,
]
assert corp_flight_bookings([[1, 2, 10], [2, 2, 15]], 2) == [10, 25]
print(corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5))

### 5.2.5 [528. Random Pick with Weight](https://leetcode.com/problems/random-pick-with-weight/) — ★★ · Medium

**Problem.** Pick index `i` with probability `w[i] / sum(w)`.  
**Signal.** Sampling by weight → prefix sums + binary search on a uniform random number.  
**Idea.** Prefix sums split `[0, total)` into intervals of length `w[i]`. Draw an integer in `[0, total)` and find the first prefix greater than it with binary search.  
**Cost.** O(n) build · O(log n) per pick

In [ ]:
class WeightedPicker:
    """Sample indices proportional to weights."""

    def __init__(self, w: list[int]) -> None:
        self.prefix: list[int] = []
        running = 0
        for x in w:
            running += x
            self.prefix.append(running)

    def pick_index(self) -> int:
        """Random index drawn by weight."""
        target = random.randrange(self.prefix[-1])
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if self.prefix[mid] > target:
                hi = mid
            else:
                lo = mid + 1
        return lo


random.seed(0)
picker = WeightedPicker([1, 3])
freq = Counter(picker.pick_index() for _ in range(10_000))
assert 0.70 < freq[1] / 10_000 < 0.80
print(freq)

---
# 6) Matrix

Matrix questions are mostly index bookkeeping. Write the boundary variables down
(`top, bottom, left, right`) and move them one at a time; for in-place updates, find a
spare place to store state (first row/column, or extra bits in each cell).

## 6.1 Traversal and in-place transforms

### 6.1.1 [54. Spiral Matrix](https://leetcode.com/problems/spiral-matrix/) — ★★★ · Medium

**Problem.** Return all elements in spiral order.  
**Signal.** Layer-by-layer walk with four shrinking boundaries.  
**Idea.** Read the top row, right column, bottom row (if still present), left column (if still present); shrink the boundaries after each side. The two "if still present" checks handle single rows/columns.  
**Cost.** O(mn) time · O(1) extra space

In [ ]:
def spiral_order(matrix: list[list[int]]) -> list[int]:
    """Elements in clockwise spiral order."""
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        out += [matrix[top][c] for c in range(left, right + 1)]
        out += [matrix[r][right] for r in range(top + 1, bottom + 1)]
        if top < bottom:
            out += [matrix[bottom][c] for c in range(right - 1, left - 1, -1)]
        if left < right:
            out += [matrix[r][left] for r in range(bottom - 1, top, -1)]
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return out


assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [
    1,
    2,
    3,
    4,
    8,
    12,
    11,
    10,
    9,
    5,
    6,
    7,
]
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]
print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

### 6.1.2 [59. Spiral Matrix II](https://leetcode.com/problems/spiral-matrix-ii/) — ★ · Medium

**Problem.** Fill an `n × n` matrix with `1..n²` in spiral order.  
**Signal.** Same boundary walk as Spiral Matrix, writing instead of reading.  
**Idea.** Walk the four sides, writing a counter; shrink boundaries after each side.  
**Cost.** O(n²) time

In [ ]:
def generate_matrix(n: int) -> list[list[int]]:
    """n x n matrix filled in spiral order."""
    grid = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    value = 1
    while top <= bottom:
        for c in range(left, right + 1):
            grid[top][c] = value
            value += 1
        for r in range(top + 1, bottom + 1):
            grid[r][right] = value
            value += 1
        for c in range(right - 1, left - 1, -1):
            if top < bottom:
                grid[bottom][c] = value
                value += 1
        for r in range(bottom - 1, top, -1):
            grid[r][left] = value
            value += 1
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return grid


assert generate_matrix(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
assert generate_matrix(1) == [[1]]
print(generate_matrix(4))

### 6.1.3 [48. Rotate Image](https://leetcode.com/problems/rotate-image/) — ★★★ · Medium

**Problem.** Rotate an `n × n` matrix 90° clockwise in place.  
**Signal.** Rotation = transpose + reverse each row.  
**Idea.** Transpose (swap `m[i][j]` with `m[j][i]` for `j > i`), then reverse each row. For counter-clockwise, reverse each row first, then transpose.  
**Cost.** O(n²) time · O(1) space

In [ ]:
def rotate_image(matrix: list[list[int]]) -> None:
    """Rotate square matrix clockwise in place."""
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()


m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_image(m)
assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print(m)

### 6.1.4 [73. Set Matrix Zeroes](https://leetcode.com/problems/set-matrix-zeroes/) — ★★★ · Medium

**Problem.** If a cell is 0, set its whole row and column to 0, in place with O(1) space.  
**Signal.** Need markers without extra memory → use the first row and column as marker storage.  
**Idea.** Record separately whether row 0 and column 0 themselves contain a zero. Mark zeros by setting `m[r][0] = m[0][c] = 0`. Then zero cells using the markers (skipping row 0 and column 0), and finally zero row 0 / column 0 if their flags are set.  
**Cost.** O(mn) time · O(1) space

In [ ]:
def set_zeroes(matrix: list[list[int]]) -> None:
    """Zero rows/columns of every zero cell, O(1) space."""
    rows, cols = len(matrix), len(matrix[0])
    first_row = any(matrix[0][c] == 0 for c in range(cols))
    first_col = any(matrix[r][0] == 0 for r in range(rows))
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][c] == 0:
                matrix[r][0] = matrix[0][c] = 0
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if first_row:
        matrix[0] = [0] * cols
    if first_col:
        for r in range(rows):
            matrix[r][0] = 0


m = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes(m)
assert m == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
m = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
set_zeroes(m)
assert m == [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
print(m)

### 6.1.5 [289. Game of Life](https://leetcode.com/problems/game-of-life/) — ★★ · Medium

**Problem.** Compute the next state of Conway's Game of Life in place.  
**Signal.** Need old and new state at once → encode both in each cell's bits.  
**Idea.** Bit 0 holds the current state, bit 1 the next state. Count live neighbours using `cell & 1`, write the next state into bit 1, then shift every cell right by one.  
**Cost.** O(mn) time · O(1) space

In [ ]:
def game_of_life(board: list[list[int]]) -> None:
    """Advance the board one generation in place."""
    rows, cols = len(board), len(board[0])
    for r in range(rows):
        for c in range(cols):
            live = sum(
                board[i][j] & 1
                for i in range(max(0, r - 1), min(rows, r + 2))
                for j in range(max(0, c - 1), min(cols, c + 2))
                if (i, j) != (r, c)
            )
            if live == 3 or (live == 2 and board[r][c] & 1):
                board[r][c] |= 2
    for r in range(rows):
        for c in range(cols):
            board[r][c] >>= 1


b = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]
game_of_life(b)
assert b == [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]
print(b)

### 6.1.6 [766. Toeplitz Matrix](https://leetcode.com/problems/toeplitz-matrix/) — ★ · Easy

**Problem.** Is every top-left to bottom-right diagonal constant?  
**Signal.** Each cell must equal its upper-left neighbour.  
**Idea.** Check `m[r][c] == m[r-1][c-1]` for all cells not in the first row/column. (Follow-up for streamed rows: only the previous row is needed.)  
**Cost.** O(mn) time · O(1) space

In [ ]:
def is_toeplitz_matrix(matrix: list[list[int]]) -> bool:
    """True if all diagonals are constant."""
    return all(
        matrix[r][c] == matrix[r - 1][c - 1]
        for r in range(1, len(matrix))
        for c in range(1, len(matrix[0]))
    )


assert is_toeplitz_matrix([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]])
assert not is_toeplitz_matrix([[1, 2], [2, 2]])
print("ok")

### 6.1.7 [498. Diagonal Traverse](https://leetcode.com/problems/diagonal-traverse/) — ★ · Medium

**Problem.** Return matrix elements in zigzag diagonal order.  
**Signal.** Cells on one anti-diagonal share `r + c`.  
**Idea.** Group cells by `d = r + c`, scanning rows top-down. Even diagonals are emitted bottom-to-top (reverse the group), odd ones top-to-bottom.  
**Cost.** O(mn) time · O(mn) space

In [ ]:
def find_diagonal_order(mat: list[list[int]]) -> list[int]:
    """Zigzag diagonal traversal."""
    groups: dict[int, list[int]] = defaultdict(list)
    for r, row in enumerate(mat):
        for c, value in enumerate(row):
            groups[r + c].append(value)
    out = []
    for d in range(len(mat) + len(mat[0]) - 1):
        out += groups[d][::-1] if d % 2 == 0 else groups[d]
    return out


assert find_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [
    1,
    2,
    4,
    7,
    5,
    3,
    6,
    8,
    9,
]
assert find_diagonal_order([[1, 2], [3, 4]]) == [1, 2, 3, 4]
print(find_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

---
# 7) Strings

String questions in interviews are mostly simulation: get every edge case (empty input,
signs, overflow, trailing spaces) right on the first pass. Say the edge cases out loud
before coding.

## 7.1 Parsing and simulation

### 7.1.1 [14. Longest Common Prefix](https://leetcode.com/problems/longest-common-prefix/) — ★★ · Easy

**Problem.** Longest common prefix of an array of strings.  
**Signal.** Vertical scan, or compare only the lexicographic min and max.  
**Idea.** After sorting, the common prefix of the whole list equals the common prefix of the first and last strings — they differ the most. `min` and `max` avoid the full sort.  
**Cost.** O(total chars) time · O(1) space

In [ ]:
def longest_common_prefix(strs: list[str]) -> str:
    """Longest prefix shared by all strings."""
    lo, hi = min(strs), max(strs)
    i = 0
    while i < len(lo) and lo[i] == hi[i]:
        i += 1
    return lo[:i]


assert longest_common_prefix(["flower", "flow", "flight"]) == "fl"
assert longest_common_prefix(["dog", "racecar", "car"]) == ""
print(longest_common_prefix(["flower", "flow", "flight"]))

### 7.1.2 [58. Length of Last Word](https://leetcode.com/problems/length-of-last-word/) — ★★ · Easy

**Problem.** Length of the last word (words separated by spaces, possibly trailing spaces).  
**Signal.** Scan from the end.  
**Idea.** Skip trailing spaces, then count characters until the next space.  
**Cost.** O(n) time · O(1) space

In [ ]:
def length_of_last_word(s: str) -> int:
    """Length of the final word."""
    i = len(s) - 1
    while s[i] == " ":
        i -= 1
    end = i
    while i >= 0 and s[i] != " ":
        i -= 1
    return end - i


assert length_of_last_word("Hello World") == 5
assert length_of_last_word("   fly me   to   the moon  ") == 4
print(length_of_last_word("luffy is still joyboy"))

### 7.1.3 [6. Zigzag Conversion](https://leetcode.com/problems/zigzag-conversion/) — ★★ · Medium

**Problem.** Write `s` in a zigzag over `num_rows` rows, then read row by row.  
**Signal.** Simulate with a row index that bounces between 0 and `num_rows − 1`.  
**Idea.** Append each character to its current row; flip direction at the top and bottom rows.  
**Cost.** O(n) time · O(n) space

In [ ]:
def convert(s: str, num_rows: int) -> str:
    """Zigzag row-by-row reading of s."""
    if num_rows == 1:
        return s
    rows = [""] * num_rows
    row, step = 0, 1
    for ch in s:
        rows[row] += ch
        if row == 0:
            step = 1
        elif row == num_rows - 1:
            step = -1
        row += step
    return "".join(rows)


assert convert("PAYPALISHIRING", 3) == "PAHNAPLSIIGYIR"
assert convert("PAYPALISHIRING", 4) == "PINALSIGYAHRPI"
assert convert("A", 1) == "A"
print(convert("PAYPALISHIRING", 3))

### 7.1.4 [8. String to Integer (atoi)](https://leetcode.com/problems/string-to-integer-atoi/) — ★★ · Medium

**Problem.** Parse an int: skip leading spaces, optional sign, digits until a non-digit, clamp to 32-bit.  
**Signal.** Step-by-step parser; clamp at the end (Python has no overflow).  
**Idea.** Skip spaces; read `+`/`−`; accumulate digits; stop at the first non-digit; clamp to `[−2³¹, 2³¹ − 1]`. In fixed-width languages check overflow before each multiply.  
**Cost.** O(n) time · O(1) space

In [ ]:
def my_atoi(s: str) -> int:
    """C-style atoi with 32-bit clamping."""
    i, n = 0, len(s)
    while i < n and s[i] == " ":
        i += 1
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    value = 0
    while i < n and s[i].isdigit():
        value = value * 10 + int(s[i])
        i += 1
    return max(-(2**31), min(2**31 - 1, sign * value))


assert my_atoi("42") == 42
assert my_atoi("   -042") == -42
assert my_atoi("1337c0d3") == 1337
assert my_atoi("words 987") == 0
assert my_atoi("-91283472332") == -(2**31)
print(my_atoi("   -042"))

### 7.1.5 [68. Text Justification](https://leetcode.com/problems/text-justification/) — ★★ · Hard

**Problem.** Fully justify words into lines of width `max_width`; last line left-justified.  
**Signal.** Greedy line packing + careful space distribution.  
**Idea.** Pack as many words as fit (each gap needs ≥ 1 space). For a normal line, distribute `max_width − letters` spaces over `gaps`; the first `extra = spaces % gaps` gaps get one more. Single-word lines and the last line are left-justified and padded on the right.  
**Cost.** O(total chars) time

In [ ]:
def full_justify(words: list[str], max_width: int) -> list[str]:
    """Fully justified lines of text."""
    lines, i = [], 0
    while i < len(words):
        j, letters = i, 0
        while j < len(words) and letters + len(words[j]) + (j - i) <= max_width:
            letters += len(words[j])
            j += 1
        gaps = j - i - 1
        if j == len(words) or gaps == 0:
            line = " ".join(words[i:j])
            lines.append(line + " " * (max_width - len(line)))
        else:
            space, extra = divmod(max_width - letters, gaps)
            line = ""
            for k in range(i, j - 1):
                line += words[k] + " " * (space + (k - i < extra))
            lines.append(line + words[j - 1])
        i = j
    return lines


words = ["This", "is", "an", "example", "of", "text", "justification."]
assert full_justify(words, 16) == [
    "This    is    an",
    "example  of text",
    "justification.  ",
]
print(full_justify(["What", "must", "be", "acknowledgment", "shall", "be"], 16))

### 7.1.6 [443. String Compression](https://leetcode.com/problems/string-compression/) — ★ · Medium

**Problem.** Compress runs in place (`aabccc` → `a2bc3`); return the new length.  
**Signal.** Run-length encoding with read and write pointers.  
**Idea.** For each run starting at `read`, find its end; write the character, then the run length's digits if the run is longer than 1.  
**Cost.** O(n) time · O(1) space

In [ ]:
def compress(chars: list[str]) -> int:
    """In-place run-length encode; return new length."""
    write = read = 0
    while read < len(chars):
        ch, start = chars[read], read
        while read < len(chars) and chars[read] == ch:
            read += 1
        chars[write] = ch
        write += 1
        if read - start > 1:
            for digit in str(read - start):
                chars[write] = digit
                write += 1
    return write


c = list("aabbccc")
k = compress(c)
assert c[:k] == list("a2b2c3")
c = list("abbbbbbbbbbbb")
assert c[: compress(c)] == list("ab12")
print(k)

### 7.1.7 [179. Largest Number](https://leetcode.com/problems/largest-number/) — ★★ · Medium

**Problem.** Arrange non-negative integers to form the largest number (as a string).  
**Signal.** Custom ordering: `a` before `b` iff `a + b > b + a` as strings.  
**Idea.** Sort with that comparator (`functools.cmp_to_key`). The comparison is transitive, so sorting is valid. Handle the all-zeros case (`"00"` → `"0"`).  
**Cost.** O(n log n · k) time for k-digit numbers

In [ ]:
def largest_number(nums: list[int]) -> str:
    """Largest concatenation of the numbers."""
    strs = [str(x) for x in nums]
    strs.sort(key=cmp_to_key(lambda a, b: -1 if a + b > b + a else 1))
    return "0" if strs[0] == "0" else "".join(strs)


assert largest_number([10, 2]) == "210"
assert largest_number([3, 30, 34, 5, 9]) == "9534330"
assert largest_number([0, 0]) == "0"
print(largest_number([3, 30, 34, 5, 9]))

### 7.1.8 [791. Custom Sort String](https://leetcode.com/problems/custom-sort-string/) — ★ · Medium

**Problem.** Permute `s` so its characters follow the order given in `order`.  
**Signal.** Counting sort with a custom key order.  
**Idea.** Count `s`; emit characters in `order` sequence (each as many times as counted), then the remaining characters in any order.  
**Cost.** O(|s| + |order|) time

In [ ]:
def custom_sort_string(order: str, s: str) -> str:
    """s rearranged to respect order."""
    count = Counter(s)
    out = [ch * count.pop(ch, 0) for ch in order]
    out += [ch * n for ch, n in count.items()]
    return "".join(out)


assert custom_sort_string("cba", "abcd") == "cbad"
assert custom_sort_string("bcafg", "abcd") == "bcad"
print(custom_sort_string("cba", "abcd"))

### 7.1.9 [408. Valid Word Abbreviation](https://leetcode.com/problems/valid-word-abbreviation/) — ★ · Easy · Premium

**Problem.** Does `abbr` (numbers replace skipped letters, no leading zeros) match `word`?  
**Signal.** Two pointers; parse multi-digit numbers in the abbreviation.  
**Idea.** Walk `abbr`: a digit run (not starting with `0`) advances the word pointer by that number; a letter must match the current word letter. Both pointers must end exactly.  
**Cost.** O(n) time · O(1) space

In [ ]:
def valid_word_abbreviation(word: str, abbr: str) -> bool:
    """True if abbr abbreviates word."""
    i = j = 0
    while j < len(abbr):
        if abbr[j].isdigit():
            if abbr[j] == "0":
                return False
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            i += num
        else:
            if i >= len(word) or word[i] != abbr[j]:
                return False
            i += 1
            j += 1
    return i == len(word)


assert valid_word_abbreviation("internationalization", "i12iz4n")
assert not valid_word_abbreviation("apple", "a2e")
assert not valid_word_abbreviation("a", "01")
print("ok")

### 7.1.10 [165. Compare Version Numbers](https://leetcode.com/problems/compare-version-numbers/) — ★ · Medium

**Problem.** Compare version strings like `1.01` vs `1.001.0`; return −1, 0 or 1.  
**Signal.** Split on dots, compare integer revisions, pad the shorter with zeros.  
**Idea.** Convert each revision to `int` (drops leading zeros) and compare position by position, treating missing revisions as 0.  
**Cost.** O(m + n) time

In [ ]:
def compare_version(version1: str, version2: str) -> int:
    """Sign of version1 - version2."""
    a = map(int, version1.split("."))
    b = map(int, version2.split("."))
    for x, y in zip_longest(a, b, fillvalue=0):
        if x != y:
            return 1 if x > y else -1
    return 0


assert compare_version("1.2", "1.10") == -1
assert compare_version("1.01", "1.001") == 0
assert compare_version("1.0", "1.0.0.0") == 0
print(compare_version("1.2", "1.10"))

### 7.1.11 [459. Repeated Substring Pattern](https://leetcode.com/problems/repeated-substring-pattern/) — ★ · Easy

**Problem.** Can `s` be built by repeating one of its substrings?  
**Signal.** Rotation trick: `s` is periodic iff it appears inside `(s + s)[1:-1]`.  
**Idea.** If `s = p·p·…·p`, rotating by `|p|` gives `s` back, so `s` occurs inside `s + s` at a position other than 0 and `|s|`. Removing the first and last characters rules out those two trivial matches.  
**Cost.** O(n) with a linear-time substring search

In [ ]:
def repeated_substring_pattern(s: str) -> bool:
    """True if s is a repetition of a shorter substring."""
    return s in (s + s)[1:-1]


assert repeated_substring_pattern("abab")
assert not repeated_substring_pattern("aba")
assert repeated_substring_pattern("abcabcabcabc")
print(repeated_substring_pattern("abab"))

---
# 8) Conclusion

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Hash map / set | pair sum, duplicates, grouping, frequency | store what you'll look up later | O(n) |
| Canonical key | anagrams, isomorphism | sorted string / count tuple / first-index pattern | O(n·k) |
| Write pointer | in-place remove/dedupe/partition | `nums[write] = x` when kept | O(n), O(1) |
| Opposite pointers | sorted pair/triple sum, palindrome, container | move the side that cannot be in the answer | O(n) |
| Sliding window | longest/shortest contiguous block, positive values | expand right, shrink left while invalid | O(n) |
| Exactly-k trick | "exactly k distinct" | `at_most(k) − at_most(k−1)` | O(n) |
| Monotonic deque | window max/min | pop dominated elements from the back | O(n) |
| Prefix sum + map | subarray sum = k / divisible, negatives allowed | look up `prefix − k` | O(n) |
| Difference array | many range updates, one read | `+v` at start, `−v` after end | O(n + q) |
| Matrix boundaries | spiral, rotate, in-place marks | four boundaries; transpose + reverse; reuse row 0 | O(mn) |